# **Install libraries**

In [3]:
!pip install roboflow
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.5 MB/s eta 0:00:00


#Get Dataset

In [1]:
import os
from dotenv import load_dotenv
from roboflow import Roboflow

load_dotenv()

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("roboflow-universe-projects").project("basketball-players-fy4c2")
version = project.version(25)
dataset = version.download("yolov12")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 58.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Basketball-Players-25 in yolov12:: 100%|██████████| 2404/2404 [00:00<00:00, 3409.04it/s]


In [9]:
import shutil
shutil.move("Basketball-Players-25/train",
            "Basketball-Players-25/Basketball-Players-25/train")

shutil.move("Basketball-Players-25/valid",
            "Basketball-Players-25/Basketball-Players-25/valid")

'Basketball-Players-25/Basketball-Players-25/valid'

#Train YOLO

In [23]:
import shutil
import os

# Ορίζουμε τις διαδρομές
source_dir = '/content/Basketball-Players-25/Basketball-Players-25'
destination_dir = '/content/Basketball-Players-25'

# Λίστα με τους φακέλους που πρέπει να μετακινήσουμε
folders_to_move = ['train', 'valid', 'test']

for folder in folders_to_move:
    src_path = os.path.join(source_dir, folder)
    dst_path = os.path.join(destination_dir, folder)

    # Ελέγχουμε αν υπάρχει ο φάκελος πριν τον μετακινήσουμε
    if os.path.exists(src_path):
        # Αν υπάρχει ήδη στον προορισμό, τον διαγράφουμε πρώτα για να μην βγάλει error
        if os.path.exists(dst_path):
            shutil.rmtree(dst_path)

        shutil.move(src_path, destination_dir)
        print(f"✅ Μετακινήθηκε ο φάκελος: {folder}")
    else:
        print(f"⚠️ Δεν βρέθηκε ο φάκελος {folder} (ίσως δεν υπάρχει στο dataset, όλα καλά)")

print("\nΗ δομή των φακέλων διορθώθηκε!")

✅ Μετακινήθηκε ο φάκελος: train
✅ Μετακινήθηκε ο φάκελος: valid
⚠️ Δεν βρέθηκε ο φάκελος test (ίσως δεν υπάρχει στο dataset, όλα καλά)

Η δομή των φακέλων διορθώθηκε!


In [24]:
import yaml

yaml_path = '/content/Basketball-Players-25/data.yaml'

data = {
    'path': '/content/Basketball-Players-25',  # Η βασική διαδρομή
    'train': 'train/images',                  # Σχετική διαδρομή
    'val': 'valid/images',                    # Σχετική διαδρομή (συνήθως το roboflow το ονομάζει valid)
    'names': {0: 'player', 1: 'referee', 2: 'ball'} # (Αυτό θα αντικατασταθεί αυτόματα αν το αρχείο υπάρχει ήδη, μην ανησυχείς)
}

# Προσπαθούμε να κρατήσουμε τα ονόματα των κλάσεων από το παλιό αρχείο αν υπάρχει
try:
    with open(yaml_path, 'r') as f:
        old_data = yaml.safe_load(f)
        if 'names' in old_data:
            data['names'] = old_data['names']
        if 'nc' in old_data:
            data['nc'] = old_data['nc']
except:
    pass

# Αποθήκευση του νέου καθαρού αρχείου
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("✅ Το data.yaml ενημερώθηκε σωστά.")

✅ Το data.yaml ενημερώθηκε σωστά.


In [25]:
!yolo task=detect mode=train model=yolo12s.pt data=/content/Basketball-Players-25/data.yaml epochs=100 imgsz=640 plots=True batch=8

Ultralytics 8.4.17 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Basketball-Players-25/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,